In [166]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [167]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

model.eval()

print("Qwen3 loaded successfully")
print("Model device:", next(model.parameters()).device)

CUDA available: True


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3 loaded successfully
Model device: cuda:0


In [168]:
def ask_qwen(prompt, max_new_tokens=500):
    messages = [
        {
            "role": "system",
            "content": "You are a query understanding module. Return only valid JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [169]:
def parse_json_response(response):
    response = response.strip()

    response = re.sub(r"```json\s*", "", response)
    response = re.sub(r"```\s*", "", response)

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in Qwen response")

    response = response[start:end + 1]

    return json.loads(response)

In [170]:
NEGATION_TRIGGERS = [
    "did not",
    "does not",
    "do not",
    "didn't",
    "doesn't",
    "don't",
    "without",
    "never",
    "but not",
    "excluding",
    "exclude",
    "not",
    "no"
]

In [171]:
def find_negation_triggers(query):
    query_lower = query.lower()
    found = []

    for trigger in NEGATION_TRIGGERS:
        if trigger in query_lower:
            found.append(trigger)

    return found

In [172]:
def verify_negation(query, structured_result):
    detected_triggers = find_negation_triggers(query)

    negation_events = {
        item.get("event_id"): item.get("trigger")
        for item in structured_result.get("negations", [])
    }

    verification_results = []

    for event in structured_result.get("events", []):
        event_id = event.get("event_id")
        qwen_negated = event.get("negated", False)

        expected_negated = event_id in negation_events
        trigger = negation_events.get(event_id)

        if qwen_negated == expected_negated:
            status = "VALID"
        else:
            status = "CORRECTED"

        if expected_negated:
            event["negated"] = True

        verification_results.append({
            "event_id": event_id,
            "action": event.get("action"),
            "original_negated": qwen_negated,
            "verified_negated": event.get("negated", False),
            "negation_trigger": trigger,
            "verification": status
        })

    if detected_triggers and not negation_events:
        verification_results.append({
            "verification": "REVIEW",
            "detected_triggers": detected_triggers,
            "message": "Negation detected in query but Qwen did not assign it to an event."
        })

    return verification_results

In [173]:
VEHICLE_KEYWORDS = {
    "car",
    "truck",
    "bus",
    "van",
    "motorcycle",
    "motorbike",
    "bicycle",
    "bike",
    "scooter",
    "vehicle"
}

TEMPORAL_TRIGGERS = [
    "then",
    "after",
    "before",
    "later",
    "followed by",
    "next",
    "finally",
    "first"
]

def verify_entities(query, structured_result):
    entities = structured_result.get("entities", [])
    verified_entities = []
    seen = set()

    for entity in entities:
        name = str(entity.get("name", "")).strip()
        entity_type = entity.get("type", "").strip().lower()

        if not name:
            continue

        key = name.lower()

        if key in seen:
            continue

        seen.add(key)

        words = key.split()

        if any(vehicle in words for vehicle in VEHICLE_KEYWORDS):
            entity_type = "vehicle"

        verified_entities.append({
            "type": entity_type,
            "name": name
        })

    structured_result["entities"] = verified_entities

    return structured_result

In [174]:
TEMPORAL_PATTERNS = [
    "followed by",
    "after that",
    "and then",
    "then",
    "later",
    "after",
    "before",
    "next",
    "finally",
    "first"
]

def verify_temporal_relations(query, structured_result):
    events = structured_result.get("events", [])

    if len(events) < 2:
        structured_result["temporal_relations"] = []
        return structured_result

    query_lower = query.lower()

    relations = []

    def add_relation(event1, event2):
        if event1 == event2:
            return

        relation = {
            "relation": "before",
            "event1": event1,
            "event2": event2
        }

        if relation not in relations:
            relations.append(relation)

    event_ids = [event["event_id"] for event in events]

    has_temporal_marker = any(
        pattern in query_lower
        for pattern in TEMPORAL_PATTERNS
    )

    if not has_temporal_marker:
        structured_result["temporal_relations"] = []
        return structured_result

    if "first" in query_lower:
        first_event = events[0]

        if len(events) > 1:
            add_relation(
                first_event["event_id"],
                events[1]["event_id"]
            )

    if "then" in query_lower or "and then" in query_lower:
        then_position = query_lower.find("then")

        before_then = []
        after_then = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < then_position:
                before_then.append(event)
            else:
                after_then.append(event)

        if before_then and after_then:
            add_relation(
                before_then[-1]["event_id"],
                after_then[0]["event_id"]
            )

    if "later" in query_lower:
        later_position = query_lower.find("later")

        before_later = []
        after_later = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < later_position:
                before_later.append(event)
            elif position > later_position:
                after_later.append(event)

        if before_later and after_later:
            add_relation(
                before_later[-1]["event_id"],
                after_later[0]["event_id"]
            )

    if "followed by" in query_lower:
        followed_position = query_lower.find("followed by")

        before_followed = []
        after_followed = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < followed_position:
                before_followed.append(event)
            elif position > followed_position:
                after_followed.append(event)

        if before_followed and after_followed:
            add_relation(
                before_followed[-1]["event_id"],
                after_followed[0]["event_id"]
            )

    if "after" in query_lower:
        after_position = query_lower.find("after")

        before_after = []
        after_after = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < after_position:
                before_after.append(event)
            elif position > after_position:
                after_after.append(event)

        if before_after and after_after:
            add_relation(
                before_after[-1]["event_id"],
                after_after[0]["event_id"]
            )

    if "before" in query_lower:
        before_position = query_lower.find("before")

        before_events = []
        after_events = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < before_position:
                before_events.append(event)
            elif position > before_position:
                after_events.append(event)

        if before_events and after_events:
            add_relation(
                after_events[0]["event_id"],
                before_events[-1]["event_id"]
            )

    if "next" in query_lower:
        next_position = query_lower.find("next")

        before_next = []
        after_next = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < next_position:
                before_next.append(event)
            elif position > next_position:
                after_next.append(event)

        if before_next and after_next:
            add_relation(
                before_next[-1]["event_id"],
                after_next[0]["event_id"]
            )

    if "finally" in query_lower:
        finally_position = query_lower.find("finally")

        before_finally = []
        after_finally = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < finally_position:
                before_finally.append(event)
            elif position > finally_position:
                after_finally.append(event)

        if before_finally and after_finally:
            add_relation(
                before_finally[-1]["event_id"],
                after_finally[0]["event_id"]
            )

    if "first" in query_lower and "then" in query_lower:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "then" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "followed by" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "finally" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    structured_result["temporal_relations"] = relations

    return structured_result

In [175]:
PROMPT_FINAL = """
You are the Query Understanding module of a video semantic search system.

Your task is to convert the user's natural-language video search query
into a structured JSON representation that will be passed to a visual
retrieval system.

You MUST extract:
1. People
2. Objects
3. Vehicles
4. Locations
5. Clothing
6. Colors and other explicitly stated attributes
7. Actions/events
8. Relationships between entities
9. Temporal relationships between events
10. Explicitly negated conditions

IMPORTANT RULES:

1. EVENT EXTRACTION
   Separate distinct actions into separate events.

2. SHARED SUBJECTS
   If multiple events refer to the same person, use the same subject
   for those events.

3. CLOTHING
   Do not create a separate event for clothing when the clothing
   describes the person performing another action.

   Example:
   "A man wearing a black jacket ate an apple."

   Correct:
   - entity: man
   - entity: black jacket
   - event: man ate apple
   - event attribute: black jacket

   Do NOT create a separate "wearing" event.

4. OBJECT ATTRIBUTES
   Preserve explicitly stated object attributes.

   Example:
   "red apple"

   Correct:
   - object: apple
   - attribute: red

5. VEHICLE ATTRIBUTES
   Preserve explicitly stated vehicle attributes.

   Example:
   "white car"

   Correct:
   - vehicle: car
   - attribute: white

6. ENTITY EXTRACTION
   Every explicitly mentioned person, object, vehicle, clothing item,
   and location must appear in the "entities" list.

7. ENTITY UNIQUENESS
   Each entity must appear only once in the "entities" list.

   Do not assign multiple types to the same entity.

8. ENTITY TYPES
   Use the most specific appropriate entity type.

   Examples:
   - "man" = person
   - "woman" = person
   - "red dress" = clothing
   - "black jacket" = clothing
   - "black bag" = object
   - "blue car" = vehicle
   - "white car" = vehicle

9. ENTITY NAMES
   Never create an entity with an empty or missing name.

10. EXPLICIT INFORMATION ONLY
    Only include entities and attributes explicitly mentioned in
    the user's query.

11. NO INVENTION
    Do not invent information.

12. NO UNSTATED INFERENCE
    Do not infer information that is not explicitly stated.

13. DESCRIPTIVE RELATIONSHIPS
    Descriptive relationships such as "wearing", "carrying",
    "holding", or "with" should normally be represented as
    attributes or relations of the relevant event rather than
    separate events.

14. TEMPORAL ORDER
    Preserve the order of events.

    Explicit temporal words include:
    "then", "after", "before", "followed by", "next",
    "first", "later", and "finally".

15. TEMPORAL RELATIONSHIPS
    Explicit temporal words must produce temporal relationships.

    Example:
    "A woman picked up a bag and then walked to a car."

    Create:
    {
        "relation": "before",
        "event1": 1,
        "event2": 2
    }

16. SEQUENTIAL EVENTS
    When multiple events occur in an explicitly stated sequence,
    create "before" relationships between the corresponding events.

17. LATER
    If the query contains "later", the event introduced after
    "later" must occur after the previously described event sequence.

18. NON-TEMPORAL CONNECTORS
    Do NOT treat "but", "however", or "although" as temporal
    relationships.

19. NEGATION
    Negation is extremely important.

    Explicit negation words and phrases include:
    "not", "no", "without", "never", "did not", "does not",
    "do not", "didn't", "doesn't", "but not", and "excluding".

20. NEGATED EVENTS
    When a condition is negated, set "negated": true for the
    corresponding event.

21. NEGATION SEPARATION
    Do NOT place a negated condition inside the attributes of
    a positive event.

22. NEGATION EXAMPLE
    Example:
    "A man ate an apple without wearing a hat."

    Correct interpretation:
    - eating apple = positive event
    - wearing hat = negative event

23. NEGATION RECORD
    Every explicit negation must also be recorded in the
    "negations" array using the corresponding event ID and
    negation trigger.

24. NO TIMESTAMPS
    Do not identify timestamps.

25. NO VISUAL RETRIEVAL
    Do not perform visual retrieval.

26. VALID JSON ONLY
    Return ONLY valid JSON.

27. JSON SYNTAX
    NEVER use invalid JSON syntax such as:

    ["color": "red"]

28. ATTRIBUTES FORMAT
    Attributes must always be a JSON array of strings.

    Example:
    ["red", "black jacket"]

29. EVENT FORMAT
    Every event must follow exactly this structure:

    {
        "event_id": 1,
        "action": "...",
        "subject": "...",
        "object": "...",
        "attributes": [],
        "relations": [],
        "negated": false
    }

30. TEMPORAL RELATION FORMAT
    Every temporal relationship must follow exactly this structure:

    {
        "relation": "before",
        "event1": 1,
        "event2": 2
    }

31. NEGATION FORMAT
    Every negation must follow exactly this structure:

    {
        "event_id": 2,
        "trigger": "without"
    }

32. VEHICLE CLASSIFICATION
    A car, truck, bus, van, motorcycle, bicycle, or other explicitly
    identified vehicle must use the entity type "vehicle".

33. COMPLETE ENTITY PRESERVATION
    Do not omit an explicitly mentioned entity just because it also
    appears inside an event.

34. ATTRIBUTE PRESERVATION
    Preserve explicitly stated visual attributes such as colors,
    clothing, object descriptions, and vehicle descriptions.

35. CONSISTENCY
    The same entity mentioned across multiple events must be
    represented consistently.

36. OUTPUT RESTRICTION
    Do not include explanations, comments, markdown, or text outside
    the JSON object.

OUTPUT FORMAT:

{
    "original_query": "...",
    "entities": [
        {
            "type": "person",
            "name": "..."
        },
        {
            "type": "clothing",
            "name": "..."
        },
        {
            "type": "object",
            "name": "..."
        },
        {
            "type": "vehicle",
            "name": "..."
        },
        {
            "type": "location",
            "name": "..."
        }
    ],
    "events": [
        {
            "event_id": 1,
            "action": "...",
            "subject": "...",
            "object": "...",
            "attributes": [],
            "relations": [],
            "negated": false
        }
    ],
    "temporal_relations": [
        {
            "relation": "before",
            "event1": 1,
            "event2": 2
        }
    ],
    "negations": [
        {
            "event_id": 2,
            "trigger": "did not"
        }
    ]
}

Return ONLY the JSON object.
"""

In [176]:
def save_structured_query(data, filename="structured_query.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"Saved to: {filename}")

In [177]:
def understand_query(query):
    prompt = PROMPT_FINAL + "\n\nUser Query:\n" + query

    response = ask_qwen(
        prompt,
        max_new_tokens=500
    )

    result = parse_json_response(response)

    result = verify_entities(
        query,
        result
    )

    result = verify_temporal_relations(
        query,
        result
    )

    verification = verify_negation(
        query,
        result
    )

    final_result = {
        "original_query": query,
        "structured_query": result,
        "negation_verification": verification
    }

    save_structured_query(final_result)

    return final_result

In [178]:
query = input("Enter video search query: ")

structured_query = understand_query(query)

print(json.dumps(structured_query, indent=4))

Enter video search query:  A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.


Saved to: structured_query.json
{
    "original_query": "A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.",
    "structured_query": {
        "original_query": "A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.",
        "entities": [
            {
                "type": "person",
                "name": "woman"
            },
            {
                "type": "person",
                "name": "man"
            },
            {
                "type": "clothing",
                "name": "red dress"
            },
            {
                "type": "clothing",
                "name": "gray jacket"
            },
            {
                "type": "object",
                "name": "black bag"
            },
            {
                "type